In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

INPUT_FILE = "INDIA_AQI_COMPLETE_20251126.csv"
OUTPUT_FILE = "mumbai_pm25_cleaned.csv"

df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("ORIGINAL DATASET")
print("=" * 60)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip().str.lower()

print("\nColumns after cleaning names:")
print(df.columns.tolist())


# ============================================================
# 3. CONVERT DATETIME
# ============================================================

df["datetime"] = pd.to_datetime(
    df["datetime"],
    errors="coerce"
)

# Remove rows where datetime could not be converted
invalid_datetime = df["datetime"].isna().sum()

print("\nInvalid datetime rows:", invalid_datetime)

df = df.dropna(subset=["datetime"])


# ============================================================
# 4. FILTER ONLY MUMBAI
# ============================================================

df["city"] = df["city"].str.strip().str.lower()

mumbai = df[df["city"] == "mumbai"].copy()

print("\n" + "=" * 60)
print("MUMBAI DATA")
print("=" * 60)

print("Mumbai rows:", len(mumbai))

print(
    "Date range:",
    mumbai["datetime"].min(),
    "to",
    mumbai["datetime"].max()
)


# ============================================================
# 5. SORT CHRONOLOGICALLY
# ============================================================

mumbai = mumbai.sort_values("datetime").reset_index(drop=True)


# ============================================================
# 6. CHECK DUPLICATE TIMESTAMPS
# ============================================================

duplicate_count = mumbai["datetime"].duplicated().sum()

print("\nDuplicate timestamps:", duplicate_count)

if duplicate_count > 0:
    print("Removing duplicate timestamps...")

    # Keep the first occurrence
    mumbai = mumbai.drop_duplicates(
        subset=["datetime"],
        keep="first"
    ).reset_index(drop=True)

print(
    "Rows after removing duplicates:",
    len(mumbai)
)


# ============================================================
# 7. REMOVE UNNECESSARY / LEAKAGE FEATURES
# ============================================================

columns_to_remove = [

    # ----------------------------------------
    # Static location information
    # ----------------------------------------
    "city",
    "state",
    "latitude",
    "longitude",

    # ----------------------------------------
    # Redundant calendar features
    # ----------------------------------------
    "day_name",
    "season",

    # ----------------------------------------
    # Redundant rain indicators
    # precipitation_mm already contains this information
    # ----------------------------------------
    "is_raining",
    "heavy_rain",

    # ----------------------------------------
    # AQI / PM2.5 derived features
    # These can cause target leakage
    # ----------------------------------------
    "us_aqi",
    "aqi_category",
    "pm25_category_india",

    # ----------------------------------------
    # Festival / crop-burning flags
    # Keep for now if you want them.
    # ----------------------------------------
]

# Only remove columns that actually exist
columns_to_remove = [
    col for col in columns_to_remove
    if col in mumbai.columns
]

print("\n" + "=" * 60)
print("REMOVING FEATURES")
print("=" * 60)

print("Columns removed:")

for col in columns_to_remove:
    print(" -", col)

mumbai = mumbai.drop(
    columns=columns_to_remove
)


# ============================================================
# 8. CHECK PM2.5
# ============================================================

print("\n" + "=" * 60)
print("PM2.5 CHECK")
print("=" * 60)

negative_pm25 = (
    mumbai["pm2_5_ugm3"] < 0
).sum()

print("Negative PM2.5 values:", negative_pm25)

# Negative PM2.5 is invalid
if negative_pm25 > 0:

    print("Converting negative PM2.5 values to NaN...")

    mumbai.loc[
        mumbai["pm2_5_ugm3"] < 0,
        "pm2_5_ugm3"
    ] = np.nan


# ============================================================
# 9. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = mumbai.isna().sum()

missing = missing[missing > 0]

if len(missing) == 0:

    print("No missing values.")

else:

    print(missing)


# ============================================================
# 10. CHECK HOURLY FREQUENCY
# ============================================================

print("\n" + "=" * 60)
print("TIMESTAMP ANALYSIS")
print("=" * 60)

time_difference = (
    mumbai["datetime"]
    .diff()
    .dropna()
)

print("\nTime difference distribution:")

print(
    time_difference
    .value_counts()
    .head(15)
)


# ============================================================
# 11. CHECK MISSING HOURLY TIMESTAMPS
# ============================================================

expected_hours = pd.date_range(
    start=mumbai["datetime"].min(),
    end=mumbai["datetime"].max(),
    freq="h"
)

actual_hours = pd.DatetimeIndex(
    mumbai["datetime"]
)

missing_hours = expected_hours.difference(
    actual_hours
)

print("\nExpected hourly timestamps:", len(expected_hours))
print("Actual timestamps:", len(actual_hours))
print("Missing timestamps:", len(missing_hours))

if len(expected_hours) > 0:

    coverage = (
        len(actual_hours)
        / len(expected_hours)
        * 100
    )

    print(
        f"Hourly coverage: {coverage:.2f}%"
    )


# ============================================================
# 12. FIND LARGEST TIME GAP
# ============================================================

max_gap = time_difference.max()

print("\nLargest timestamp gap:", max_gap)

print(
    "Largest gap in hours:",
    max_gap.total_seconds() / 3600
)


# ============================================================
# 13. CREATE CONTINUOUS SEGMENTS
# ============================================================

# A new segment starts whenever the difference
# between two timestamps is NOT exactly one hour.

mumbai["time_diff"] = (
    mumbai["datetime"]
    .diff()
)

mumbai["segment"] = (
    mumbai["time_diff"]
    != pd.Timedelta(hours=1)
).cumsum()


# ============================================================
# 14. ANALYZE CONTINUOUS SEGMENTS
# ============================================================

segment_sizes = (
    mumbai
    .groupby("segment")
    .size()
)

print("\n" + "=" * 60)
print("CONTINUOUS SEGMENTS")
print("=" * 60)

print(
    "Number of continuous segments:",
    len(segment_sizes)
)

print(
    "Longest continuous segment:",
    segment_sizes.max(),
    "hours"
)

print(
    "Segments >= 24 hours:",
    (segment_sizes >= 24).sum()
)

print(
    "Segments >= 48 hours:",
    (segment_sizes >= 48).sum()
)

print(
    "Segments >= 96 hours:",
    (segment_sizes >= 96).sum()
)


# ============================================================
# 15. REMOVE TEMPORARY TIME DIFFERENCE COLUMN
# ============================================================

mumbai = mumbai.drop(
    columns=["time_diff"]
)


# ============================================================
# 16. DISPLAY FINAL FEATURES
# ============================================================

print("\n" + "=" * 60)
print("FINAL DATASET")
print("=" * 60)

print("Final shape:", mumbai.shape)

print("\nFinal columns:")

for i, col in enumerate(mumbai.columns, 1):
    print(f"{i}. {col}")


# ============================================================
# 17. DISPLAY FIRST 5 ROWS
# ============================================================

print("\nFirst 5 rows:")

print(
    mumbai.head().to_string()
)


# ============================================================
# 18. SAVE CLEANED DATASET
# ============================================================

mumbai.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

print(
    f"Cleaned Mumbai dataset saved as: {OUTPUT_FILE}"
)

ORIGINAL DATASET
Shape: (842160, 71)

Columns:
['City', 'State', 'Latitude', 'Longitude', 'Datetime', 'Year', 'Month', 'Day', 'Hour', 'Day_of_Week', 'Day_Name', 'Week_of_Year', 'Is_Weekend', 'Quarter', 'Season', 'Time_of_Day', 'Temp_2m_C', 'Temp_80m_C', 'Temp_120m_C', 'Temp_180m_C', 'Humidity_Percent', 'Dew_Point_C', 'Humidity_Category', 'Wind_Speed_10m_kmh', 'Wind_Speed_80m_kmh', 'Wind_Speed_120m_kmh', 'Wind_Dir_10m', 'Wind_Gusts_kmh', 'Wind_Category', 'Wind_Stagnation', 'Precipitation_mm', 'Rain_mm', 'Is_Raining', 'Heavy_Rain', 'Pressure_MSL_hPa', 'Surface_Pressure_hPa', 'Solar_Radiation_Wm2', 'Direct_Radiation_Wm2', 'Diffuse_Radiation_Wm2', 'UV_Index', 'Cloud_Cover_Percent', 'Cloud_Low_Percent', 'Cloud_Mid_Percent', 'Cloud_High_Percent', 'Is_Daytime', 'Sunshine_Seconds', 'PM2_5_ugm3', 'PM10_ugm3', 'PM_Ratio', 'CO_ugm3', 'NO2_ugm3', 'SO2_ugm3', 'O3_ugm3', 'Dust_ugm3', 'NH3_ugm3', 'AOD', 'US_AQI', 'US_AQI_PM25', 'US_AQI_PM10', 'US_AQI_NO2', 'US_AQI_O3', 'US_AQI_CO', 'EU_AQI', 'EU_AQI_